# PROJECT : Trading Game #2 — Optimized 1-vs-1 Pairs Trading

Commodity: Sugar (G18)

**Strategy**: Exploit mean reversion between Sugar Futures and the most cointegrated equity  
**Features**: Dynamic lookback Z-score · Volatility filter · Adaptive trailing stop-loss  
**Metrics**: Sharpe Ratio · Max Drawdown · Calmar Ratio

---

## Common Part :
Demonstration of how we imported our data and build our dataset : **sugar_dataset_G18**

We commented the code so we won't be bothered by it later

## 0. Install & Import

In [ ]:
# Install required packages
#!pip install yfinance statsmodels -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from itertools import combinations

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('All packages loaded successfully.')

## 1. Data Collection

**Universe (G18 — Sugar)**:
- **Anchor**: Sugar #11 Futures (`SB=F`)
- **Sector ETFs**: `SGG` (iPath Sugar ETN), `MOO` (Agribusiness ETF)
- **Individual equities**: CSAN, BG, ADM, TR, HSY, MDLZ, SRE, WILMAR (OTC: `WLMIY`), TATE (`TATE.L`), ABF (`ABF.L`)

In [ ]:
# ============================================================
# ASSET UNIVERSE — G18 Sugar
# ============================================================

ANCHOR = 'SB=F'   # Sugar futures

ETFS = {
    'SGG': 'iPath Bloomberg Sugar ETN',
    'MOO': 'VanEck Agribusiness ETF',
    'DBA': 'Invesco DB Agriculture ETF',
    'CANE': 'Teucrium Sugar Fund',   # optional, may fail on Yahoo depending on availability
}

EQUITIES = {
    'CSAN': 'Cosan SA',
    'BG':   'Bunge Global',
    'ADM':  'Archer-Daniels-Midland',
    'TR':   'Tootsie Roll Industries',
    'HSY':  'Hershey Company',
    'MDLZ': 'Mondelez International',
    'SJM':  'J.M. Smucker',
    'KHC':  'Kraft Heinz',
    'CPB':  'Campbell Soup',
    'PEP':  'PepsiCo',
    'KDP':  'Keurig Dr Pepper',
    'GIS':  'General Mills',
    'CAG':  'Conagra Brands',
    'MKC':  'McCormick',
    'KO':   'Coca-Cola',
}

START_DATE = '2019-01-01'
END_DATE   = '2024-12-31'

ALL_TICKERS = [ANCHOR] + list(ETFS.keys()) + list(EQUITIES.keys())

print(f"Total tickers requested: {len(ALL_TICKERS)}")
print(ALL_TICKERS)

In [ ]:
# # downloading data
# print("\nDownloading OHLCV data from Yahoo Finance...")
# raw = yf.download(
#     ALL_TICKERS,
#     start=START_DATE,
#     end=END_DATE,
#     auto_adjust=True,
#     progress=True
# )

# prices = raw['Close'].copy()


# # Drop very incomplete columns
# threshold = 0.20  # max 20% missing values
# missing_pct = prices.isna().mean()
# dropped = missing_pct[missing_pct > threshold].index.tolist()

# if dropped:
#     print(f"\nDropped tickers with >20% missing data: {dropped}")

# prices = prices.drop(columns=dropped)
# prices = prices.ffill().dropna()


# # Check what survived
# retained = prices.columns.tolist()
# retained_anchor = [t for t in [ANCHOR] if t in retained]
# retained_etfs = [t for t in ETFS.keys() if t in retained]
# retained_equities = [t for t in EQUITIES.keys() if t in retained]

# print("\n=== DATASET SUMMARY ===")
# print(f"Dataset shape: {prices.shape}")
# print(f"Date range: {prices.index[0].date()} -> {prices.index[-1].date()}")
# print(f"Retained total assets: {len(retained)}")
# print(f"Retained anchor ({len(retained_anchor)}): {retained_anchor}")
# print(f"Retained ETFs ({len(retained_etfs)}): {retained_etfs}")
# print(f"Retained equities ({len(retained_equities)}): {retained_equities}")

# if len(retained_equities) < 10:
#     print("\nWARNING: fewer than 10 retained stocks -> risk of penalty.")
# if len(retained) < 15:
#     print("\nWARNING: fewer than 15 retained assets total -> consider adding more backup tickers.")


# # Save cleaned price panel
# prices.to_csv("sugar_prices_panel.csv")
# print("\nSaved cleaned price panel to sugar_prices_panel.csv")

# # Save full OHLCV dataset
# ohlcv_frames = []

# for ticker in retained:
#     try:
#         df_t = raw.xs(ticker, axis=1, level=1).copy()
#         df_t.columns = [f"{ticker}_{col}" for col in df_t.columns]
#         ohlcv_frames.append(df_t)
#     except Exception as e:
#         print(f"Could not extract OHLCV for {ticker}: {e}")

# full_dataset = pd.concat(ohlcv_frames, axis=1).ffill()
# full_dataset.to_csv("sugar_dataset_G18.csv")

# print("Saved full OHLCV dataset to sugar_dataset_G18.csv")
# print(f"Full OHLCV shape: {full_dataset.shape}")

Here upload the dataset : **sugar_dataset_G18**

In [ ]:
from google.colab import files
uploaded = files.upload()

import io
filename = list(uploaded.keys())[0]  # prend le nom automatiquement peu importe le nom
prices = pd.read_csv(io.BytesIO(uploaded[filename]),
                     index_col=0, parse_dates=True)

# Keep only Close prices
close_cols = [col for col in prices.columns if 'Close' in col]
prices = prices[close_cols]

# Clean column names (remove 'Close_' prefix if present)
prices.columns = [col.replace('Close_', '').replace('_Close', '') for col in prices.columns]

print(prices.shape)
print(prices.columns.tolist())

# Trading Game #2 Part

# 1. Cointegration Analysis

We test cointegration between the **Sugar Futures anchor** and each individual equity.  
We select the pair with the **lowest p-value** (Engle-Granger test).

In [ ]:
# Stationarity & Cointegration : We want to find which stock moves most with sugar futures


import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import warnings
warnings.filterwarnings('ignore')

# log prices are more stable for these tests
log_prices = np.log(prices)
anchor = 'SB=F'

# ADF test : We need all series to be I(1) before running cointegration
def adf_test(series, name):
    p_level = adfuller(series.dropna(), autolag='AIC')[1]
    p_diff  = adfuller(series.dropna().diff().dropna(), autolag='AIC')[1]
    return {
        'ticker'  : name,
        'p_level' : round(p_level, 4),
        'p_diff'  : round(p_diff,  4),
        'is_I1'   : (p_level > 0.05) and (p_diff < 0.05)
    }

adf_results = pd.DataFrame([adf_test(log_prices[col], col) for col in log_prices.columns])
print("ADF results:")
print(adf_results.to_string(index=False))

# keep only I(1) series
i1 = adf_results[adf_results['is_I1'] == True]['ticker'].tolist()
candidates = [t for t in i1 if t != anchor]
print(f"\nI(1) series : {i1}")
print(f"Testing cointegration against {anchor} for : {candidates}")


# Engle-Granger between SB=F and each candidate
anchor_s = log_prices[anchor]
eg_results = []

for ticker in candidates:
    s = log_prices[ticker]
    idx = anchor_s.index.intersection(s.index)
    _, pval, _ = coint(anchor_s[idx], s[idx])
    eg_results.append({
        'ticker' : ticker,
        'pvalue' : round(pval, 4),
        'coint'  : pval < 0.05
    })

eg_df = pd.DataFrame(eg_results).sort_values('pvalue').reset_index(drop=True)
print("\nEngle-Granger results (sorted by p-value):")
print(eg_df.to_string(index=False))

# best pair = lowest p-value
BEST_PAIR = eg_df.iloc[0]['ticker']
coint_5pct = eg_df[eg_df['coint'] == True]['ticker'].tolist()

print(f"\nBest pair with {anchor} : {BEST_PAIR}")
print(f"Cointegrated at 5%    : {coint_5pct}")


# Johansen on the top pairs to confirm

top = coint_5pct if coint_5pct else eg_df.head(3)['ticker'].tolist()
top = top[:4]  # keep it manageable

if len(top) >= 2:
    cols     = [anchor] + top
    joh_data = log_prices[cols].dropna()
    joh      = coint_johansen(joh_data, det_order=0, k_ar_diff=1)

    print(f"\nJohansen test on {cols}")
    print("\nTrace stat:")
    for k in range(len(joh.lr1)):
        sig = "reject H0" if joh.lr1[k] > joh.cvt[k, 1] else "fail to reject"
        print(f"  r={k} : stat={joh.lr1[k]:.3f}  cv95={joh.cvt[k,1]:.3f}  -> {sig}")

    print("\nMax eigenvalue stat:")
    for k in range(len(joh.lr2)):
        sig = "reject H0" if joh.lr2[k] > joh.cvm[k, 1] else "fail to reject"
        print(f"  r={k} : stat={joh.lr2[k]:.3f}  cv95={joh.cvm[k,1]:.3f}  -> {sig}")
else:
    print("\nNot enough cointegrated pairs for Johansen, sticking with EG result.")

print(f"\n=> BEST_PAIR = '{BEST_PAIR}' — will be used for the Z-score strategy")

In [ ]:
import matplotlib.pyplot as plt
# VISUALISE THE PAIR (consistent with cointegration)

EQUITY_TICKER = BEST_PAIR

p1 = log_prices[anchor]
p2 = log_prices[EQUITY_TICKER]

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(p1 / p1.iloc[0] * 100, label='Sugar Futures (SB=F)', lw=1.5)
ax.plot(p2 / p2.iloc[0] * 100, label=EQUITY_TICKER, lw=1.5)

ax.set_title(f'Normalised Log Prices: SB=F vs {EQUITY_TICKER}')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('pair_normalised_prices.png', dpi=150)
plt.show()

## 2. Spread & Dynamic Z-Score

We estimate the **hedge ratio** via OLS rolling regression, then compute the **spread**.  
The Z-score is computed with a **dynamic lookback** optimised on a rolling basis.

In [ ]:
# Z-score with dynamic lookback : We trade the spread between SB=F and MDLZ

from scipy.stats import zscore
from sklearn.linear_model import LinearRegression

# Compute the spread (OLS hedge ratio, rolling)
# We don't use a fixed beta — we re-estimate it every day on a rolling window to stay adaptive

s1 = log_prices[anchor]       # SB=F
s2 = log_prices[BEST_PAIR]    # MDLZ

# candidate lookback windows to test (in trading days)
lookback_candidates = [30, 60, 90, 120, 180]

def compute_spread(s1, s2, window):
    """Rolling OLS: regress s1 on s2, extract residuals as spread."""
    spread = pd.Series(index=s1.index, dtype=float)
    for i in range(window, len(s1)):
        y = s1.iloc[i-window:i].values
        x = s2.iloc[i-window:i].values.reshape(-1, 1)
        beta = LinearRegression().fit(x, y).coef_[0]
        intercept = LinearRegression().fit(x, y).intercept_
        spread.iloc[i] = s1.iloc[i] - beta * s2.iloc[i] - intercept
    return spread

def compute_zscore(spread, window):
    """Rolling z-score of the spread."""
    mu  = spread.rolling(window).mean()
    sig = spread.rolling(window).std()
    return (spread - mu) / sig

# Dynamic lookback selection
# For each candidate window, we compute the spread + z-score and pick the window that maximizes spread mean-reversion

from statsmodels.tsa.stattools import adfuller

best_window = None
best_pval   = 1.0
results_lb  = []

for w in lookback_candidates:
    spread = compute_spread(s1, s2, w).dropna()
    pval   = adfuller(spread, autolag='AIC')[1]
    results_lb.append({'window': w, 'adf_pval': round(pval, 4)})
    if pval < best_pval:
        best_pval   = pval
        best_window = w

print("Lookback window selection (lower ADF p-value = more stationary spread):")
print(pd.DataFrame(results_lb).to_string(index=False))
print(f"\nBest window selected : {best_window} days (ADF p-value = {best_pval})")

# Compute final spread and z-score with best window

spread  = compute_spread(s1, s2, best_window).dropna()
z_score = compute_zscore(spread, best_window).dropna()

# align everything on the same index
common_idx = spread.index.intersection(z_score.index)
spread  = spread[common_idx]
z_score = z_score[common_idx]

print(f"\nSpread shape  : {spread.shape}")
print(f"Z-score range : {z_score.min():.2f} to {z_score.max():.2f}")
print(f"Z-score mean  : {z_score.mean():.4f}  (should be ~0)")
print(f"Z-score std   : {z_score.std():.4f}   (should be ~1)")

#Quick plot to sanity check

import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(log_prices[anchor][common_idx],    label='SB=F (log)',  color='royalblue')
axes[0].plot(log_prices[BEST_PAIR][common_idx], label='MDLZ (log)', color='tomato', alpha=0.8)
axes[0].set_title('Log prices — SB=F vs MDLZ')
axes[0].legend()

axes[1].plot(spread, color='seagreen', linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title(f'Rolling spread (window={best_window}d)')

axes[2].plot(z_score, color='darkorange', linewidth=0.8)
axes[2].axhline( 2,  color='red',   linestyle='--', linewidth=0.8, label='+2 / -2 (entry)')
axes[2].axhline(-2,  color='red',   linestyle='--', linewidth=0.8)
axes[2].axhline( 0,  color='black', linestyle='-',  linewidth=0.5, label='exit')
axes[2].axhline( 3,  color='darkred', linestyle=':',linewidth=0.7, label='+3 / -3 (stop)')
axes[2].axhline(-3,  color='darkred', linestyle=':',linewidth=0.7)
axes[2].set_title(f'Z-score (window={best_window}d)')
axes[2].legend()

plt.tight_layout()
plt.savefig('zscore_spread.png', dpi=150)
plt.show()
print("Plot saved as zscore_spread.png")

## 3. Trading Strategy

**Entry rules**:  
- Z > +2 → Short spread (short Sugar, long equity)  
- Z < -2 → Long spread (long Sugar, short equity)  

**Exit rules**:  
- Z crosses back toward 0 (±1 threshold)  

**Volatility filter**: No new entries when 20d realised vol of spread > 1.5× its long-run average  

**Stop-loss**: Adaptive trailing stop — exit if spread moves 3σ against position

In [ ]:
# Volatility Filter

returns_anchor = log_prices[anchor].diff().dropna()

vol_window    = 20
rolling_vol   = returns_anchor.rolling(vol_window).std()
vol_threshold = rolling_vol.quantile(0.75)

vol_filter  = rolling_vol < vol_threshold

# align with z_score index
common_idx  = z_score.index.intersection(vol_filter.index)
z_score     = z_score[common_idx]
spread      = spread[common_idx]
vol_filter  = vol_filter[common_idx]
rolling_vol = rolling_vol[common_idx]

print(f"Vol threshold (75th pct) : {vol_threshold:.5f}")
print(f"Days allowed  : {vol_filter.sum()} ({vol_filter.mean()*100:.1f}%)")
print(f"Days blocked  : {(~vol_filter).sum()} ({(~vol_filter).mean()*100:.1f}%)")

In [ ]:
# BACKTEST — adaptive trailing stop (version corrigée)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRAILING_DISTANCE = 1.5
ENTRY_Z = 2.0
EXIT_Z = 0.0
TRANSACTION_COST = 0.0010  # 10 bps

# =========================
# 1) DataFrame
# =========================
df_bt = pd.DataFrame(index=common_idx)
df_bt["spread"] = spread
df_bt["z"] = z_score
df_bt["vol_filter"] = vol_filter.astype(bool)

# =========================
# 2) Backtest positions
# =========================
position = 0
best_z = None
positions = []
trail_levels = []

for z, can_trade in zip(df_bt["z"], df_bt["vol_filter"]):
    trailing_stop = np.nan

    if position == 0:
        if can_trade:
            if z < -ENTRY_Z:
                position = 1      # long spread
                best_z = z
            elif z > ENTRY_Z:
                position = -1     # short spread
                best_z = z

    elif position == 1:
        # long spread: favorable move = z becomes even lower
        best_z = min(best_z, z)
        trailing_stop = best_z + TRAILING_DISTANCE

        # exit on mean reversion or trailing stop
        if z >= EXIT_Z or z >= trailing_stop:
            position = 0
            best_z = None
            trailing_stop = np.nan

    elif position == -1:
        # short spread: favorable move = z becomes even higher
        best_z = max(best_z, z)
        trailing_stop = best_z - TRAILING_DISTANCE

        # exit on mean reversion or trailing stop
        if z <= EXIT_Z or z <= trailing_stop:
            position = 0
            best_z = None
            trailing_stop = np.nan

    positions.append(position)
    trail_levels.append(trailing_stop)

df_bt["position"] = positions
df_bt["trailing_stop"] = trail_levels

# =========================
# 3) Returns
# =========================
df_bt["spread_ret"] = df_bt["spread"].diff().fillna(0)
df_bt["pos_shift"] = df_bt["position"].shift(1).fillna(0)
df_bt["trade"] = df_bt["position"].diff().abs().fillna(0)

df_bt["strategy_ret"] = (
    df_bt["pos_shift"] * df_bt["spread_ret"]
    - df_bt["trade"] * TRANSACTION_COST
)

# Ne PAS faire dropna() sur tout le DataFrame
# On garde toutes les dates
df_bt["cum_ret"] = df_bt["strategy_ret"].cumsum()
df_bt["rolling_max"] = df_bt["cum_ret"].cummax()
df_bt["drawdown"] = df_bt["cum_ret"] - df_bt["rolling_max"]

# =========================
# 4) Global metrics
# =========================
ret_mean = df_bt["strategy_ret"].mean()
ret_std = df_bt["strategy_ret"].std()

sharpe = (ret_mean / ret_std) * np.sqrt(252) if ret_std != 0 else np.nan
max_dd = df_bt["drawdown"].min()
total_return = df_bt["cum_ret"].iloc[-1]
calmar = total_return / abs(max_dd) if max_dd != 0 else np.nan

entries = ((df_bt["position"].shift(1).fillna(0) == 0) & (df_bt["position"] != 0))
n_trades = int(entries.sum())
days_traded = int((df_bt["position"] != 0).sum())

# =========================
# 5) True win rate by completed trade
# =========================
trade_pnls = []
in_trade = False
current_trade_pnl = 0.0

for i in range(len(df_bt)):
    prev_pos = df_bt["position"].iloc[i - 1] if i > 0 else 0
    curr_pos = df_bt["position"].iloc[i]
    day_ret = df_bt["strategy_ret"].iloc[i]

    # entry
    if not in_trade and prev_pos == 0 and curr_pos != 0:
        in_trade = True
        current_trade_pnl = day_ret

    # trade ongoing
    elif in_trade:
        current_trade_pnl += day_ret

        # exit
        if curr_pos == 0:
            trade_pnls.append(current_trade_pnl)
            in_trade = False
            current_trade_pnl = 0.0

# Optional: include open trade at the end
if in_trade:
    trade_pnls.append(current_trade_pnl)

win_rate = np.mean(np.array(trade_pnls) > 0) if len(trade_pnls) > 0 else np.nan
avg_trade_pnl = np.mean(trade_pnls) if len(trade_pnls) > 0 else np.nan

# =========================
# 6) Performance summary
# =========================
print("=" * 40)
print("PERFORMANCE SUMMARY")
print("=" * 40)
print(f"Total return : {total_return:.3f}")
print(f"Sharpe Ratio : {sharpe:.3f}")
print(f"Max Drawdown : {max_dd:.3f}")
print(f"Calmar Ratio : {calmar:.3f}")
print(f"Nb trades    : {n_trades}")
print(f"Win rate     : {win_rate*100:.1f}%")
print(f"Avg trade PnL: {avg_trade_pnl:.4f}")
print(f"Days traded  : {days_traded} / {len(df_bt)}")

# =========================
# 7) Plots
# =========================
plt.figure(figsize=(12, 5))
plt.plot(df_bt.index, df_bt["cum_ret"], label="Cumulative Strategy Return")
plt.title("Strategy Performance")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(df_bt.index, df_bt["z"], label="Z-score")
plt.axhline(ENTRY_Z, linestyle="--", label="+Entry")
plt.axhline(-ENTRY_Z, linestyle="--", label="-Entry")
plt.axhline(EXIT_Z, linestyle=":", label="Mean Reversion Exit")
plt.axhline(0, linestyle=":")
plt.title("Z-score Signal")
plt.xlabel("Date")
plt.ylabel("Z-score")
plt.legend()
plt.grid(True)
plt.show()

## 4. Results

In [ ]:
summary = pd.Series({
    'Pair'              : f'{anchor} vs {BEST_PAIR}',
    'Lookback window'   : best_window,
    'Total return'      : round(total_return, 3),
    'Sharpe ratio'      : round(sharpe, 3),
    'Max drawdown'      : round(max_dd, 3),
    'Calmar ratio'      : round(calmar, 3),
    'Number of trades'  : n_trades,
    'Win rate'          : f'{win_rate*100:.1f}%',
    'Days in market'    : int(df_bt['position'].astype(bool).sum()),
})

print("\n=== FINAL STRATEGY RESULTS ===")
print(summary.to_string())

# save for report
summary.to_csv("game2_final_results.csv", header=True)
print("\nSaved as game2_final_results.csv")